# PHQ-8 Depression Detection - Classical ML Baselines

This notebook builds participant-level classical machine learning baselines from the precomputed log-mel spectrograms.

**Design goals**
- Reuse the existing `processed/spectrograms` data.
- Keep a shared train/dev/test pipeline across models.
- Store each trained model under `best_model/<model-name>/`.
- Aggregate segment-level spectrograms into participant-level tabular features.


## 1. Install Dependencies

Run this cell if your notebook kernel does not already have the required packages.


In [ ]:
%pip install -q numpy pandas scikit-learn openpyxl joblib matplotlib

## 2. Imports And Configuration

In [1]:
from __future__ import annotations

import json
from abc import ABC, abstractmethod
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import ElasticNet, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR

BASE_DIR = Path('.')
SPECTROGRAM_ROOT = BASE_DIR / 'processed' / 'spectrograms'
METADATA_XLSX = SPECTROGRAM_ROOT / 'segment_metadata.xlsx'
MODEL_ROOT = BASE_DIR / 'best_model'
MODEL_ROOT.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
SPLITS = ('train', 'dev', 'test')

print(f'Spectrogram root: {SPECTROGRAM_ROOT.resolve()}')
print(f'Metadata file   : {METADATA_XLSX.resolve()}')
print(f'Model output dir: {MODEL_ROOT.resolve()}')
if not METADATA_XLSX.exists():
    raise FileNotFoundError(f'Metadata spreadsheet not found: {METADATA_XLSX}')

Spectrogram root: /home/redstonerosh12/Documents/dev/dl/Project/sutd_50.039_Deep_Learning/processed/spectrograms
Metadata file   : /home/redstonerosh12/Documents/dev/dl/Project/sutd_50.039_Deep_Learning/processed/spectrograms/segment_metadata.xlsx
Model output dir: /home/redstonerosh12/Documents/dev/dl/Project/sutd_50.039_Deep_Learning/best_model


## 3. Load Metadata

The provided spreadsheet stores one row per segment with participant labels and split membership.


In [2]:
metadata = pd.read_excel(METADATA_XLSX)

numeric_columns = [
    'participant_id', 'segment_idx', 'n_mels', 'n_frames',
    'participant_dur_s', 'n_turns', 'phq_score', 'binary_label'
]
for col in numeric_columns:
    metadata[col] = pd.to_numeric(metadata[col], errors='coerce')

metadata['participant_id'] = metadata['participant_id'].astype(int)
metadata['segment_idx'] = metadata['segment_idx'].astype(int)
metadata['split'] = metadata['split'].astype(str).str.strip().str.lower()
metadata['spectrogram_path'] = metadata['spectrogram_path'].astype(str)

def resolve_spectrogram_path(path_str: str) -> Path:
    cleaned = path_str.replace('\\', '/').strip()
    candidate = BASE_DIR / cleaned
    if candidate.exists():
        return candidate
    fallback = SPECTROGRAM_ROOT / Path(cleaned).name
    if fallback.exists():
        return fallback
    raise FileNotFoundError(f'Could not resolve spectrogram path: {path_str}')

metadata['resolved_path'] = metadata['spectrogram_path'].map(resolve_spectrogram_path)

display(metadata.head())
print(metadata['split'].value_counts().to_string())
print('\nParticipants per split:')
print(metadata.groupby('split')['participant_id'].nunique().to_string())

,participant_id,segment_idx,spectrogram_path,n_mels,n_frames,participant_dur_s,n_turns,phq_score,binary_label,split,resolved_path
0,300,0,processed\spectrograms\test\300_seg0000.npy,128,801,155.76,87,2,0,test,processed/spectrograms/test/300_seg0000.npy
1,300,1,processed\spectrograms\test\300_seg0001.npy,128,801,155.76,87,2,0,test,processed/spectrograms/test/300_seg0001.npy
2,300,2,processed\spectrograms\test\300_seg0002.npy,128,801,155.76,87,2,0,test,processed/spectrograms/test/300_seg0002.npy
3,300,3,processed\spectrograms\test\300_seg0003.npy,128,801,155.76,87,2,0,test,processed/spectrograms/test/300_seg0003.npy
4,300,4,processed\spectrograms\test\300_seg0004.npy,128,801,155.76,87,2,0,test,processed/spectrograms/test/300_seg0004.npy


split
train    11362
test      5904
dev       4322

Participants per split:
split
dev       35
test      47
train    107


## 4. Segment Feature Extraction

Each spectrogram is converted into a compact numeric descriptor. We then aggregate these segment features to the participant level.


In [3]:
def extract_segment_features(spec: np.ndarray) -> np.ndarray:
    spec = np.asarray(spec, dtype=np.float32)

    mel_mean = spec.mean(axis=1)
    mel_std = spec.std(axis=1)

    time_delta = np.diff(spec, axis=1)
    if time_delta.size == 0:
        flux_mean = np.zeros(spec.shape[0], dtype=np.float32)
        flux_std = np.zeros(spec.shape[0], dtype=np.float32)
    else:
        flux_mean = np.mean(np.abs(time_delta), axis=1)
        flux_std = np.std(time_delta, axis=1)

    global_stats = np.array([
        spec.mean(),
        spec.std(),
        spec.min(),
        spec.max(),
        np.mean(np.abs(time_delta)) if time_delta.size else 0.0,
        np.std(time_delta) if time_delta.size else 0.0,
    ], dtype=np.float32)

    return np.concatenate([mel_mean, mel_std, flux_mean, flux_std, global_stats]).astype(np.float32)

sample_spec = np.load(metadata.iloc[0]['resolved_path'])
sample_feat = extract_segment_features(sample_spec)
print('Sample spectrogram shape:', sample_spec.shape)
print('Segment feature size   :', sample_feat.shape[0])

Sample spectrogram shape: (128, 801)
Segment feature size   : 518


## 5. Participant-Level Aggregation

The label belongs to a participant, not to an individual segment. To avoid segment-level leakage, we aggregate features per participant before training.


In [4]:
participant_rows = []

for participant_id, group in metadata.groupby('participant_id', sort=True):
    segment_features = []
    for path in group['resolved_path']:
        spec = np.load(path)
        segment_features.append(extract_segment_features(spec))

    segment_features = np.vstack(segment_features)

    feature_vector = np.concatenate([
        segment_features.mean(axis=0),
        segment_features.std(axis=0),
        np.array([
            len(segment_features),
            float(group['participant_dur_s'].iloc[0]),
            float(group['n_turns'].iloc[0]),
        ], dtype=np.float32),
    ]).astype(np.float32)

    participant_rows.append({
        'participant_id': int(participant_id),
        'split': str(group['split'].iloc[0]),
        'phq_score': float(group['phq_score'].iloc[0]),
        'binary_label': int(group['binary_label'].iloc[0]),
        'n_segments': int(len(segment_features)),
        'feature_vector': feature_vector,
    })

participants_df = pd.DataFrame(participant_rows)

X_train = np.vstack(participants_df.loc[participants_df['split'] == 'train', 'feature_vector'].to_list())
y_train = participants_df.loc[participants_df['split'] == 'train', 'phq_score'].to_numpy(dtype=np.float32)

X_dev = np.vstack(participants_df.loc[participants_df['split'] == 'dev', 'feature_vector'].to_list())
y_dev = participants_df.loc[participants_df['split'] == 'dev', 'phq_score'].to_numpy(dtype=np.float32)

X_test = np.vstack(participants_df.loc[participants_df['split'] == 'test', 'feature_vector'].to_list())
y_test = participants_df.loc[participants_df['split'] == 'test', 'phq_score'].to_numpy(dtype=np.float32)

split_participants = {}
for split_name in SPLITS:
    split_participants[split_name] = participants_df.loc[
        participants_df['split'] == split_name, 'participant_id'
    ].to_numpy(dtype=int)

split_features = {
    'train': (X_train, y_train),
    'dev': (X_dev, y_dev),
    'test': (X_test, y_test),
}

print('Participant dataframe shape:', participants_df.shape)
print('Feature dimension         :', X_train.shape[1])
for split_name in SPLITS:
    X_split, y_split = split_features[split_name]
    print(f'{split_name:5s}: X={X_split.shape}, y={y_split.shape}, PHQ mean={y_split.mean():.2f}')

Participant dataframe shape: (189, 6)
Feature dimension         : 1039
train: X=(107, 1039), y=(107,), PHQ mean=6.42
dev  : X=(35, 1039), y=(35,), PHQ mean=7.43
test : X=(47, 1039), y=(47,), PHQ mean=6.98


## 6. Shared Model Interface

Each model wrapper follows the same API. `forward()` acts as the prediction step, while `loss()` computes regression metrics.


In [5]:
def pearson_correlation(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    if y_true.size < 2:
        return 0.0
    if np.std(y_true) == 0.0 or np.std(y_pred) == 0.0:
        return 0.0
    return float(np.corrcoef(y_true, y_pred)[0, 1])

class BaseRegressor(ABC):
    def __init__(self, name: str):
        self.name = name
        self.pipeline = self.build_pipeline()

    @abstractmethod
    def build_pipeline(self) -> Pipeline:
        raise NotImplementedError

    def fit(self, X: np.ndarray, y: np.ndarray):
        self.pipeline.fit(X, y)
        return self

    def forward(self, X: np.ndarray) -> np.ndarray:
        return self.pipeline.predict(X)

    def loss(self, y_true: np.ndarray, y_pred: np.ndarray) -> dict:
        mse = mean_squared_error(y_true, y_pred)
        rmse = float(np.sqrt(mse))
        mae = mean_absolute_error(y_true, y_pred)
        r = pearson_correlation(y_true, y_pred)
        return {
            'mse': float(mse),
            'rmse': float(rmse),
            'mae': float(mae),
            'pearson_r': float(r),
        }

    def evaluate(self, X: np.ndarray, y: np.ndarray) -> tuple[np.ndarray, dict]:
        preds = self.forward(X)
        metrics = self.loss(y, preds)
        return preds, metrics

    def save(self, output_dir: Path, metrics: dict, split_predictions: dict, split_participant_ids: dict):
        output_dir.mkdir(parents=True, exist_ok=True)
        joblib.dump(self.pipeline, output_dir / 'model.joblib')

        with (output_dir / 'metrics.json').open('w', encoding='utf-8') as f:
            json.dump(metrics, f, indent=2)

        records = []
        for split_name, preds in split_predictions.items():
            ids = split_participant_ids[split_name]
            for participant_id, pred in zip(ids, preds):
                records.append({
                    'split': split_name,
                    'participant_id': int(participant_id),
                    'predicted_phq_score': float(pred),
                })

        pd.DataFrame(records).to_csv(output_dir / 'predictions.csv', index=False)

class RidgeRegressor(BaseRegressor):
    def build_pipeline(self) -> Pipeline:
        return Pipeline([
            ('scaler', StandardScaler()),
            ('pca', PCA(n_components=0.95, svd_solver='full', random_state=RANDOM_STATE)),
            ('model', Ridge(alpha=10.0)),
        ])

class ElasticNetRegressor(BaseRegressor):
    def build_pipeline(self) -> Pipeline:
        return Pipeline([
            ('scaler', StandardScaler()),
            ('pca', PCA(n_components=0.95, svd_solver='full', random_state=RANDOM_STATE)),
            ('model', ElasticNet(alpha=0.05, l1_ratio=0.3, max_iter=10000, random_state=RANDOM_STATE)),
        ])

class SVRRegressor(BaseRegressor):
    def build_pipeline(self) -> Pipeline:
        return Pipeline([
            ('scaler', StandardScaler()),
            ('pca', PCA(n_components=0.95, svd_solver='full', random_state=RANDOM_STATE)),
            ('model', SVR(kernel='rbf', C=10.0, epsilon=0.5)),
        ])

class RandomForestBaseline(BaseRegressor):
    def build_pipeline(self) -> Pipeline:
        return Pipeline([
            ('model', RandomForestRegressor(
                n_estimators=400,
                max_depth=None,
                min_samples_leaf=2,
                random_state=RANDOM_STATE,
                n_jobs=-1,
            )),
        ])

models = {
    'ridge': RidgeRegressor('ridge'),
    'elastic_net': ElasticNetRegressor('elastic_net'),
    'svr': SVRRegressor('svr'),
    'random_forest': RandomForestBaseline('random_forest'),
}

print('Models:', ', '.join(models.keys()))

Models: ridge, elastic_net, svr, random_forest


## 7. Train, Evaluate, And Save Models

Classical models are trained once on the participant-level train split. We evaluate each model on train, dev, and test, then save the fitted pipeline and predictions.


In [6]:
results = []
trained_models = {}

for model_name, model in models.items():
    print(f'\n=== Training {model_name} ===')
    model.fit(X_train, y_train)

    split_metrics = {}
    split_predictions = {}

    for split_name in SPLITS:
        X_split, y_split = split_features[split_name]
        preds, metrics = model.evaluate(X_split, y_split)
        split_metrics[split_name] = metrics
        split_predictions[split_name] = preds

        row = {
            'model': model_name,
            'split': split_name,
            **metrics,
        }
        results.append(row)
        print(
            f"{split_name:5s} | MSE={metrics['mse']:.4f} | RMSE={metrics['rmse']:.4f} "
            f"| MAE={metrics['mae']:.4f} | r={metrics['pearson_r']:.4f}"
        )

    output_dir = MODEL_ROOT / model_name
    model.save(
        output_dir=output_dir,
        metrics=split_metrics,
        split_predictions=split_predictions,
        split_participant_ids=split_participants,
    )
    trained_models[model_name] = model
    print(f'Saved artifacts to: {output_dir.resolve()}')

results_df = pd.DataFrame(results)
results_df


=== Training ridge ===
train | MSE=13.9814 | RMSE=3.7392 | MAE=2.8189 | r=0.7262
dev   | MSE=51.3802 | RMSE=7.1680 | MAE=5.7561 | r=0.0015
test  | MSE=4608647680.0000 | RMSE=67887.0214 | MAE=9908.2881 | r=0.1607
Saved artifacts to: /home/redstonerosh12/Documents/dev/dl/Project/sutd_50.039_Deep_Learning/best_model/ridge

=== Training elastic_net ===
train | MSE=13.9789 | RMSE=3.7388 | MAE=2.8212 | r=0.7262
dev   | MSE=51.5838 | RMSE=7.1822 | MAE=5.7678 | r=-0.0022
test  | MSE=4798708224.0000 | RMSE=69272.7091 | MAE=10110.4209 | r=0.1607
Saved artifacts to: /home/redstonerosh12/Documents/dev/dl/Project/sutd_50.039_Deep_Learning/best_model/elastic_net

=== Training svr ===
train | MSE=6.9902 | RMSE=2.6439 | MAE=1.4483 | r=0.8950
dev   | MSE=53.5833 | RMSE=7.3201 | MAE=5.9532 | r=-0.1622
test  | MSE=44.7037 | RMSE=6.6861 | MAE=5.2903 | r=0.0451
Saved artifacts to: /home/redstonerosh12/Documents/dev/dl/Project/sutd_50.039_Deep_Learning/best_model/svr

=== Training random_forest ===
train |

,model,split,mse,rmse,mae,pearson_r
0,ridge,train,1.398135e+01,3.739165,2.818910,0.726185
1,ridge,dev,5.138023e+01,7.168000,5.756055,0.001454
2,ridge,test,4.608648e+09,67887.021440,9908.288086,0.160690
3,elastic_net,train,1.397892e+01,3.738839,2.821192,0.726203
4,elastic_net,dev,5.158384e+01,7.182189,5.767848,-0.002180
5,elastic_net,test,4.798708e+09,69272.709085,10110.420898,0.160690
6,svr,train,6.990220e+00,2.643902,1.448300,0.895022
7,svr,dev,5.358335e+01,7.320065,5.953247,-0.162154
8,svr,test,4.470371e+01,6.686083,5.290332,0.045112
9,random_forest,train,4.564330e+00,2.136429,1.685927,0.983743


## 8. Compare Dev And Test Performance

In [7]:
summary = results_df.pivot(index='model', columns='split', values=['rmse', 'mae', 'pearson_r'])
summary = summary.sort_values(('rmse', 'dev'))
summary

rmse                               mae                \
split               dev          test     train       dev          test   
model                                                                     
random_forest  6.612033      6.491837  2.136429  5.521914      5.387562   
ridge          7.168000  67887.021440  3.739165  5.756055   9908.288086   
elastic_net    7.182189  69272.709085  3.738839  5.767848  10110.420898   
svr            7.320065      6.686083  2.643902  5.953247      5.290332   

                        pearson_r                      
split             train       dev      test     train  
model                                                  
random_forest  1.685927  0.072323  0.054052  0.983743  
ridge          2.818910  0.001454  0.160690  0.726185  
elastic_net    2.821192 -0.002180  0.160690  0.726203  
svr            1.448300 -0.162154  0.045112  0.895022

## 9. Select The Best Model By Dev RMSE

In [8]:
dev_results = results_df[results_df['split'] == 'dev'].sort_values('rmse')
best_model_name = dev_results.iloc[0]['model']
best_model_dir = MODEL_ROOT / best_model_name

print(f'Best model by dev RMSE: {best_model_name}')
print(f'Artifacts saved in    : {best_model_dir.resolve()}')
display(dev_results)

Best model by dev RMSE: random_forest
Artifacts saved in    : /home/redstonerosh12/Documents/dev/dl/Project/sutd_50.039_Deep_Learning/best_model/random_forest


,model,split,mse,rmse,mae,pearson_r
10,random_forest,dev,43.718974,6.612033,5.521914,0.072323
1,ridge,dev,51.380230,7.168000,5.756055,0.001454
4,elastic_net,dev,51.583843,7.182189,5.767848,-0.002180
7,svr,dev,53.583349,7.320065,5.953247,-0.162154
